# Portfolio Project 1: Customer Churn Classification

> Follow a compact end-to-end workflow from raw customer table to an evaluated and saved model.

**Track:** Machine Learning Roadmap · **Language:** English

## Project brief
**Goal:** predict whether a customer will churn using a small synthetic telecom dataset. This data is generated for teaching, not collected from real customers.

You will inspect data quality, define the target, build a leakage-safe preprocessing pipeline, compare a baseline with a tree-based model, evaluate class-specific performance, and save the model artifact. Keep business interpretation cautious: a predictive association does not prove that an intervention will prevent churn.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

csv_path = Path("../datasets/telecom_churn_synthetic.csv")
if not csv_path.exists():
    csv_path = Path("datasets/telecom_churn_synthetic.csv")
df = pd.read_csv(csv_path)
print(df.shape)
display(df.head())
print("Missing values:")
print(df.isna().sum())
print("Target rate:", round(df["churn"].mean(),3))

(720, 7)
Missing values:
customer_id         0
age                11
tenure_months       0
monthly_charges    13
contract_type       8
support_plan        0
churn               0
dtype: int64
Target rate: 0.219


,customer_id,age,tenure_months,monthly_charges,contract_type,support_plan,churn
0,C00000,67.0,40,109.82,one-year,no,0
1,C00001,28.0,48,79.14,month-to-month,yes,0
2,C00002,19.0,18,74.67,month-to-month,no,1
3,C00003,55.0,13,18.52,month-to-month,yes,0
4,C00004,39.0,23,67.77,two-year,no,0


In [2]:
# The customer identifier is not a predictive feature.
target = "churn"
feature_cols = ["age", "tenure_months", "monthly_charges", "contract_type", "support_plan"]
X, y = df[feature_cols], df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, stratify=y, random_state=42)
num = ["age", "tenure_months", "monthly_charges"]
cat = ["contract_type", "support_plan"]
pre = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat),
])
models = {
    "Dummy baseline": Pipeline([("pre", pre), ("model", DummyClassifier(strategy="most_frequent"))]),
    "Logistic regression": Pipeline([("pre", pre), ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))]),
    "Random forest": Pipeline([("pre", pre), ("model", RandomForestClassifier(n_estimators=160, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=1))]),
}
results=[]
for name, model in models.items():
    model.fit(X_train,y_train)
    p = model.predict_proba(X_test)[:,1]
    results.append({"model":name,"ROC-AUC":roc_auc_score(y_test,p),"PR-AUC/AP":average_precision_score(y_test,p)})
display(pd.DataFrame(results).sort_values("PR-AUC/AP",ascending=False).round(3))

,model,ROC-AUC,PR-AUC/AP
2,Random forest,0.698,0.474
1,Logistic regression,0.708,0.461
0,Dummy baseline,0.500,0.222


## Evaluate the selected model without overstating the result
For a portfolio, select a candidate based on validation/CV performance and operational criteria. The example below is illustrative and fits one candidate on the training partition. In a real project, use cross-validation and document the selection rule before the final test evaluation.

In [3]:
final_model = models["Logistic regression"]
final_model.fit(X_train, y_train)
prob = final_model.predict_proba(X_test)[:,1]
pred = (prob >= .5).astype(int)
print(classification_report(y_test,pred,zero_division=0))
print("Test ROC-AUC:", round(roc_auc_score(y_test,prob),3))

              precision    recall  f1-score   support

           0       0.85      0.64      0.73       112
           1       0.32      0.59      0.42        32

    accuracy                           0.63       144
   macro avg       0.58      0.62      0.57       144
weighted avg       0.73      0.63      0.66       144

Test ROC-AUC: 0.708


In [4]:
import joblib, json, tempfile
from pathlib import Path
with tempfile.TemporaryDirectory() as tmp:
    artifact = Path(tmp) / "churn_pipeline.joblib"
    joblib.dump(final_model, artifact)
    restored = joblib.load(artifact)
    example = X_test.head(3)
    assert np.allclose(restored.predict_proba(example), final_model.predict_proba(example))
    print("Artifact saved and round-trip checked:", artifact.name)

project_summary = {
    "dataset": "telecom_churn_synthetic.csv (synthetic teaching data)",
    "target": "churn within the simulated labeling process",
    "features": feature_cols,
    "split_seed": 42,
    "limitations": ["Synthetic relationships", "Not evidence of real customer behavior", "No causal claims"]
}
print(json.dumps(project_summary, indent=2))

Artifact saved and round-trip checked: churn_pipeline.joblib
{
  "dataset": "telecom_churn_synthetic.csv (synthetic teaching data)",
  "target": "churn within the simulated labeling process",
  "features": [
    "age",
    "tenure_months",
    "monthly_charges",
    "contract_type",
    "support_plan"
  ],
  "split_seed": 42,
  "limitations": [
    "Synthetic relationships",
    "Not evidence of real customer behavior",
    "No causal claims"
  ]
}


## Portfolio checklist
- State the business question and prediction time.
- Explain target and feature definitions.
- Show data quality decisions and split strategy.
- Compare against a baseline using relevant metrics.
- Describe limitations, fairness/privacy considerations, and deployment assumptions.
- Include reproducible setup instructions and do not commit sensitive or real customer data.